# Bobby Carrot RL Training (Colab T4 GPU)

This notebook trains the Bobby Carrot agent for a SINGLE specific level using Behavioral Cloning (BC) pre-training followed by PPO fine-tuning.

In [ ]:
!pip install pygame torch numpy
# Set headless dummy display for PyGame
import os
os.environ["SDL_VIDEODRIVER"] = "dummy"

In [ ]:
import sys
import os

# Add Game_Python to path assuming you've cloned the repository here
repo_path = "/content/RL_Game_Project"
if os.path.exists(repo_path):
    sys.path.insert(0, repo_path)
    sys.path.insert(0, os.path.join(repo_path, "Game_Python"))
else:
    print("Please mount Google Drive or clone the repository into /content/RL_Game_Project")

In [ ]:
from Bobby_Carrot.rl_models.config import LevelConfig, PPOConfig, TrainingConfig
from Bobby_Carrot.rl_models.ppo import train_ppo
from Bobby_Carrot.rl_models.expert_data import EXPERT
from pathlib import Path

################################################
# 1. SET THE SPECIFIC LEVEL YOU WANT TO TRAIN
################################################
LEVEL_KIND = "normal"
LEVEL_NUMBER = 1  # Change this to 2, 3, etc., to train different levels separately

# Training Configuration
ppo_config = PPOConfig(
    lr=3e-4,
    clip_ratio=0.2,
    n_epochs=4,
    rollout_length=4096,
    minibatch_size=128,
    entropy_coeff=0.15,
    entropy_min=0.08
)

train_config = TrainingConfig(
    total_timesteps=100000,  # Modify per-level timesteps here
    device="cuda",
    checkpoint_dir=Path(f"/content/checkpoints/{LEVEL_KIND}_{LEVEL_NUMBER}"),
    log_dir=Path("/content/logs"),
    curriculum=False,
    max_steps_per_episode=1500
)

print(f"\n=== Starting Training for Level {LEVEL_KIND} {LEVEL_NUMBER} ===")
single_level_config = LevelConfig(train_levels=[(LEVEL_KIND, LEVEL_NUMBER)], test_levels=[(LEVEL_KIND, LEVEL_NUMBER)])

# Run the training. It will automatically use the correct expert array for this specific level if available.
train_ppo(ppo_config, train_config, single_level_config, expert_moves=EXPERT.get(LEVEL_NUMBER, []))
print(f"=== Finished Training for Level {LEVEL_KIND} {LEVEL_NUMBER} ===")
